# Linear types, code generation, and scheduling in Alpha

This notebook separates two questions that are easy to conflate:

1. **Linear validity:** does each linear resource point flow exactly once through the source program?
2. **Schedule validity:** does a target mapping assign every statement instance a unique execution time and preserve every data dependence?

Alpha checks linearity before lowering and scheduling. A schedule may reorder independent work, but it cannot repair an invalid resource flow or change which points are consumed.

## Configure the notebook

Import the installed `alphalang` extension. The examples are deterministic and need no external data or random state.

In [1]:
import alphalang

## Define the task

A linear declaration attaches multiplicity to every point in a variable's polyhedral domain. This transfer consumes each `X[i]` once and produces each `Y[i]` once. The Python binding exposes immutable snapshots of that resolved metadata.

In [2]:
LINEAR_TRANSFER = """affine Transfer [N] -> {: N > 0}
    inputs linear X : {[i] : 0 <= i < N};
    outputs linear Y : {[i] : 0 <= i < N};
    let Y[i] = X[i];
.
"""

DUPLICATE_USE = """affine Duplicate [N] -> {: N > 0}
    inputs linear X : {[i] : 0 <= i < N};
    outputs linear Y : {[i] : 0 <= i < N};
            linear Z : {[i] : 0 <= i < N};
    let Y[i] = X[i]; Z[i] = X[i];
.
"""

PARTIAL_USE = """affine Partial [N] -> {: N > 1}
    inputs linear X : {[i] : 0 <= i < N};
    outputs linear Y : {[i] : 0 <= i < N - 1};
    let Y[i] = X[i];
.
"""

BROADCAST_USE = """affine Broadcast [N] -> {: N > 1}
    inputs linear X : {[i] : 0 <= i < N};
    outputs linear Y : {[i] : 0 <= i < N};
    let Y[i] = X[0];
.
"""

## Implement reusable inspection helpers

The compiler may report more than one consequence of an invalid flow. For a compact tutorial, `first_diagnostic` prints the first semantic diagnostic after the count header.

In [3]:
def first_diagnostic(label, source):
    try:
        alphalang.parse(source)
    except ValueError as error:
        lines = str(error).splitlines()
        print(f"{label}: {lines[1] if len(lines) > 1 else lines[0]}")


def variable_summary(variables):
    return tuple(
        (variable.name, repr(variable.multiplicity), variable.domain)
        for variable in variables
    )

## Run the linear examples

First inspect the accepted transfer. `inputs`, `outputs`, and `locals` are tuples of immutable `Variable` snapshots; multiplicity and resolved ISL domains remain available after parsing.

In [4]:
transfer = alphalang.parse(LINEAR_TRANSFER)
print("inputs:", variable_summary(transfer.inputs))
print("outputs:", variable_summary(transfer.outputs))
print("locals:", transfer.locals)

inputs: (('X', 'Multiplicity.LINEAR', '[N] -> { [i] : 0 <= i < N }'),)
outputs: (('Y', 'Multiplicity.LINEAR', '[N] -> { [i] : 0 <= i < N }'),)
locals: ()


### Exact-once failures

These examples fail for different relational reasons:

- **Duplicate:** the same `X[i]` points feed two outputs, so use ranges overlap.
- **Partial:** the final point of `X` is never consumed.
- **Broadcast:** many `Y[i]` instances read `X[0]`, so the access relation is not injective.

The checks operate on exact ISL maps and sets, not occurrence counts.

In [5]:
first_diagnostic("duplicate", DUPLICATE_USE)
first_diagnostic("partial", PARTIAL_USE)
first_diagnostic("broadcast", BROADCAST_USE)

duplicate: uses of linear variable 'X' overlap on [N] -> { [i0] : 0 <= i0 < N }
partial: linear variable 'X' has unconsumed points: [N] -> { [i = -1 + N] : N >= 2 }
broadcast: one use of linear variable 'X' consumes points more than once: [N] -> { [i] -> [0] : N >= 2 and 0 <= i < N }


### Explicit call signatures

Opaque operations must declare what they do with linear values. A linear input port promises to consume its argument exactly once; output multiplicity is independent of input multiplicity. Legacy declarations such as `external f(1)` remain unrestricted.

In [6]:
EXTERNAL_MOVE = "external move(linear) -> linear\n" + LINEAR_TRANSFER.replace(
    "X[i]", "move(X[i])"
)
moved = alphalang.parse(EXTERNAL_MOVE)
print(moved.outputs[0].multiplicity)

Multiplicity.LINEAR


## Scheduling and code generation

A valid schedule must:

- cover every statement instance (totality),
- map distinct instances to distinct times (injectivity),
- use compatible schedule-space widths,
- refer only to known statement names, and
- place every producer strictly before each consumer.

The pointwise transfer has no dependence between different `i` instances. Both identity order and reverse order are legal, and both preserve the already-checked relation `Y[i] <- X[i]`.

In [7]:
normalized = alphalang.normalize(transfer)
identity = normalized.schedule("")
reversed_order = normalized.schedule("[N] -> { Y[i] -> [N - 1 - i]; }")
print("identity: legal")
print("reverse: legal")

code = alphalang.generate(reversed_order)
print("generated C:", "#include" in code and "Y" in code)

identity: legal
reverse: legal
generated C: True


### Invalid target mapping

`Y[i] -> [0]` collapses every instance onto one timestamp. It is rejected as non-injective before dependence legality is considered.

In [8]:
try:
    normalized.schedule("[N] -> { Y[i] -> [0]; }")
except alphalang.ScheduleError as error:
    print(f"ScheduleError: {error}")

ScheduleError: target mapping for statement 'Y' is not injective — two instances would land on the same schedule-space point, making their relative order undefined (§6)


### Legal shape, illegal dependence order

The next source is linearly valid: `X[i]` flows once into local `T[i]`, then `T[i]` flows once into output `Y[i]`. Its schedule must still respect the true dependence from statement `T` to statement `Y`.

In [9]:
PRODUCER_CONSUMER = """affine Pipeline [N] -> {: N > 0}
    inputs linear X : {[i] : 0 <= i < N};
    outputs linear Y : {[i] : 0 <= i < N};
    locals linear T : {[i] : 0 <= i < N};
    let T[i] = X[i]; Y[i] = T[i];
.
"""

pipeline = alphalang.normalize(alphalang.parse(PRODUCER_CONSUMER))
legal_pipeline = pipeline.schedule(
    "[N] -> { T[i] -> [0, i]; Y[i] -> [1, i]; }"
)
print("producer before consumer: legal")

try:
    pipeline.schedule("[N] -> { T[i] -> [1, i]; Y[i] -> [0, i]; }")
except alphalang.ScheduleError as error:
    print(f"ScheduleError: {error}")

producer before consumer: legal
ScheduleError: 'Y' reads 'T' but the schedule doesn't guarantee the producer instance runs strictly before the consumer instance that reads it (§7.2)


## Validate the results

The important separation is now visible:

- **Linear validity** constrains pointwise resource flow in the source and is checked before scheduling.
- **Mapping validity** requires total, injective, consistently shaped schedules over known statements.
- **Dependence legality** requires scheduled producer times to be strictly earlier than consumer times.

Identity and reversal are both legal for independent transfer instances. Reversal is not legal when it places `Y[i]` before the `T[i]` value that `Y[i]` reads.

In [10]:
assert transfer.inputs[0].multiplicity == alphalang.Multiplicity.LINEAR
assert transfer.outputs[0].multiplicity == alphalang.Multiplicity.LINEAR
assert normalized.inputs[0].multiplicity == alphalang.Multiplicity.LINEAR
assert "#include" in code
assert isinstance(identity, alphalang.ScheduledSystem)
assert isinstance(reversed_order, alphalang.ScheduledSystem)
assert isinstance(legal_pipeline, alphalang.ScheduledSystem)
print("all checks passed")

all checks passed
